# 05 — Menambah Data dari Sumber Non-BBO (PBN Championship Archives)

**Latar belakang**: diminta membuat crawler BBO skala besar (target >=1 juta
board). Dicek dulu ToS BBO (`news.bridgebase.com/terms`) — eksplisit melarang
scraping/automation software. Crawler BBO TIDAK dibuat.

Sebagai gantinya, dicari sumber publik non-BBO. Dua kandidat awal
(`bridgetoernooi.com`, `sarantakos.com/bridge/vugraph.html`) ternyata sudah
**mati** (domain expired/di-squat) meski muncul di hasil pencarian. Yang
terverifikasi masih hidup: **computerbridge.se** — arsip PBN 7 final
kejuaraan dunia (Bermuda Bowl 2015/2017/2019, Venice Cup, Vanderbilt,
Spingold, Soloway KO, Swedish Cup 2019), format bebas non-komersial.

Selain itu, ditemukan **100 file LIN baru** muncul di `data/raw/` selama
sesi ini (kemungkinan besar hasil export manual "My Hands" pribadi Anda
dari BBO — itu legitimate, beda dari automated scraping) — jadi dataset
kanonik sendiri sudah bertambah dari 506 -> 606 file sebelum PBN
ditambahkan.

**Fitur baru**: `src/parser/pbn_parser.py` (`PBNParser`) — parser PBN yang
menghasilkan `BoardRecord`/`Hand` yang sama persis dengan `LINParser`,
supaya `extract_features()` dan seluruh pipeline jalan tanpa perubahan.
`build_dataset()` diperluas dengan parameter `extra_boards` untuk
menggabungkan board dari sumber manapun.


In [1]:
import sys
import time
from pathlib import Path

import pandas as pd
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

PROJECT_ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.parser import PBNParser
from src.preprocessing.dataset_builder import load_splits
from src.evaluation.metrics import evaluate, print_summary, compare_models

OUT_DIR = PROJECT_ROOT / "experiments" / "2026-07-15" / "outputs" / "combined_data"
OUT_DIR.mkdir(parents=True, exist_ok=True)


## 1. Verifikasi sumber PBN

7 file ZIP dari computerbridge.se sudah diunduh & diekstrak ke
`data/raw_pbn/` (43 file .pbn). Parser diverifikasi bekerja sebelum
dipakai untuk membangun dataset (lihat langkah download & verifikasi
di riwayat kerja — tidak diulang di sini karena sudah tersimpan).


In [2]:
pbn_boards = PBNParser().parse_directory(PROJECT_ROOT / "data" / "raw_pbn")
print(f"Total board dari 7 arsip kejuaraan (PBN): {len(pbn_boards)}")

from collections import Counter
event_cnt = Counter(b.source_file.rsplit("_", 1)[0] for b in pbn_boards)
for k, v in sorted(event_cnt.items()):
    print(f"  {k}: {v} board")


Total board dari 7 arsip kejuaraan (PBN): 1314
  bermuda_bowl_2015: 256 board
  bermuda_bowl_2017: 192 board
  bermuda_bowl_2019: 192 board
  soloway_ko_2019: 119 board
  spingold_2019: 120 board
  swedish_cup_2019: 123 board
  vanderbilt_2019: 120 board
  venice_cup_2019: 192 board


## 2. Dataset gabungan

`data/processed_combined/` = `data/raw/` (606 file LIN, termasuk 100 file
baru) + `data/raw_pbn/` (43 file PBN, 1.314 board) + 18 fitur DDS
(cache diperluas mencakup semua board baru). Dibangun lewat
`build_dataset(extra_boards=pbn_boards, include_dds=True, ...)` —
lihat riwayat kerja untuk perintah build lengkapnya.


In [3]:
df_train, df_val, df_test, feature_cols, le = load_splits(PROJECT_ROOT / "data" / "processed_combined")
X_train, y_train = df_train[feature_cols].values, df_train["label"].values
X_test, y_test = df_test[feature_cols].values, df_test["label"].values

print(f"Train: {X_train.shape}, Val: {df_val.shape}, Test: {X_test.shape}")
print(f"Total board: {len(df_train)+len(df_val)+len(df_test)} (naik dari 10.223)")
print(f"Fitur: {len(feature_cols)} (164 kanonik + 18 DDS)")
print(f"Kelas: {len(le.classes_)}")


Train: (9508, 182), Val: (2037, 192), Test: (2036, 182)
Total board: 13581 (naik dari 10.223)
Fitur: 182 (164 kanonik + 18 DDS)
Kelas: 35


## 3. Latih ulang kandidat terbaik di data yang diperluas

Dua kandidat terbaik sejauh ini: XGBoost acc-tuned (F1 weighted tinggi,
accuracy tertinggi single-model) dan LightGBM+class_weight (F1 macro
terbaik). Dilatih ulang di data gabungan, dievaluasi di test set.


In [4]:
XGB_ACC_TUNED = dict(
    n_estimators=300, max_depth=5, learning_rate=0.03,
    subsample=0.9, colsample_bytree=0.6, min_child_weight=5, reg_lambda=2.0,
    random_state=42, n_jobs=-1, eval_metric="mlogloss", verbosity=0,
)
LGBM_CLASS_WEIGHT = dict(
    n_estimators=300, max_depth=-1, learning_rate=0.05, num_leaves=63,
    subsample=0.8, colsample_bytree=0.8, random_state=42, n_jobs=-1,
    verbose=-1, class_weight="balanced",
)

results = []

t0 = time.time()
clf_xgb = XGBClassifier(**XGB_ACC_TUNED)
clf_xgb.fit(X_train, y_train)
proba = clf_xgb.predict_proba(X_test)
res = evaluate(y_test, proba.argmax(axis=1), proba, le, model_name="XGBoost acc-tuned (data gabungan)")
res["fit_seconds"] = round(time.time() - t0, 1)
results.append(res)
print_summary(res)

t0 = time.time()
clf_lgbm = LGBMClassifier(**LGBM_CLASS_WEIGHT)
clf_lgbm.fit(X_train, y_train)
proba2 = clf_lgbm.predict_proba(X_test)
res2 = evaluate(y_test, proba2.argmax(axis=1), proba2, le, model_name="LightGBM class_weight (data gabungan)")
res2["fit_seconds"] = round(time.time() - t0, 1)
results.append(res2)
print_summary(res2)



  XGBoost acc-tuned (data gabungan)
  Accuracy          : 0.5236
  Precision (macro) : 0.3308
  Recall (macro)    : 0.2487
  F1 (macro)        : 0.2638
  F1 (weighted)     : 0.4841
  Top-3 Accuracy    : 0.7844
  Top-5 Accuracy    : 0.8689


d:\Bridge-Prediction\.venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



  LightGBM class_weight (data gabungan)
  Accuracy          : 0.5363
  Precision (macro) : 0.4066
  Recall (macro)    : 0.2852
  F1 (macro)        : 0.3100
  F1 (weighted)     : 0.5063
  Top-3 Accuracy    : 0.7785
  Top-5 Accuracy    : 0.8536


## 4. Bandingkan dengan baseline 10.223-board (sebelum data tambahan)

In [5]:
comparison = compare_models(results)
comparison.to_csv(OUT_DIR / "test_comparison.csv")

# Angka referensi dari notebooks_dds/04_evaluation.ipynb (10.223 board, sebelum data tambahan)
reference = {
    "XGBoost acc-tuned (10.223 board, REF)": {
        "accuracy": 0.5270711024135681, "f1_macro": 0.25071198222661645, "f1_weighted": 0.4802467915918208,
    },
    "LightGBM class_weight (10.223 board, REF)": {
        "accuracy": 0.5257664709719504, "f1_macro": 0.3067643072391079, "f1_weighted": 0.4854029125273938,
    },
}

print("Perbandingan sebelum vs sesudah data tambahan:")
for r in results:
    ref_key = f"{r['model'].split(' (')[0]} (10.223 board, REF)"
    if ref_key in reference:
        ref = reference[ref_key]
        print(f"  {r['model']}")
        print(f"    accuracy:    {ref['accuracy']:.4f} -> {r['accuracy']:.4f}  ({(r['accuracy']-ref['accuracy'])*100:+.2f}pp)")
        print(f"    f1_macro:    {ref['f1_macro']:.4f} -> {r['f1_macro']:.4f}  ({(r['f1_macro']-ref['f1_macro'])*100:+.2f}pp)")
        print(f"    f1_weighted: {ref['f1_weighted']:.4f} -> {r['f1_weighted']:.4f}  ({(r['f1_weighted']-ref['f1_weighted'])*100:+.2f}pp)")

comparison


Perbandingan sebelum vs sesudah data tambahan:
  XGBoost acc-tuned (data gabungan)
    accuracy:    0.5271 -> 0.5236  (-0.35pp)
    f1_macro:    0.2507 -> 0.2638  (+1.31pp)
    f1_weighted: 0.4802 -> 0.4841  (+0.38pp)
  LightGBM class_weight (data gabungan)
    accuracy:    0.5258 -> 0.5363  (+1.06pp)
    f1_macro:    0.3068 -> 0.3100  (+0.32pp)
    f1_weighted: 0.4854 -> 0.5063  (+2.09pp)


,accuracy,precision_macro,recall_macro,f1_macro,f1_weighted,top_3_accuracy,top_5_accuracy
model,,,,,,,
XGBoost acc-tuned (data gabungan),0.523576,0.330848,0.248652,0.263789,0.484055,0.784381,0.868861
LightGBM class_weight (data gabungan),0.536346,0.406630,0.285150,0.310001,0.506285,0.778487,0.853635


In [6]:
import json

summary = {
    "generated": pd.Timestamp.now().isoformat(),
    "data_sources": {
        "lin_files_original": 506,
        "lin_files_new_today": 100,
        "lin_files_total": 606,
        "pbn_files": 43,
        "pbn_boards": len(pbn_boards),
    },
    "dataset_size": {
        "train": len(df_train), "val": len(df_val), "test": len(df_test),
        "total": len(df_train) + len(df_val) + len(df_test),
    },
    "results": {r["model"]: {k: r[k] for k in
        ["accuracy", "f1_macro", "f1_weighted", "top_3_accuracy", "top_5_accuracy"]}
        for r in results},
    "reference_10223_boards": reference,
}
(OUT_DIR / "summary.json").write_text(json.dumps(summary, indent=2))
print(f"Saved: {OUT_DIR / 'summary.json'}")


Saved: d:\Bridge-Prediction\experiments\2026-07-15\outputs\combined_data\summary.json


## 5. Kesimpulan

**Hasil terbaik baru dari seluruh proyek** — data tambahan (10.223 -> 13.582
board, +33%) memberi perbaikan nyata, terutama untuk LightGBM:

| Model | Accuracy | F1 Macro | F1 Weighted |
|---|---|---|---|
| XGBoost acc-tuned (10.223 board) | 52.7% | 0.251 | 0.480 |
| XGBoost acc-tuned (13.582 board) | 52.4% (-0.3pp) | 0.264 (+1.3pp) | 0.484 (+0.4pp) |
| LightGBM class_weight (10.223 board) | 52.6% | 0.307 | 0.485 |
| **LightGBM class_weight (13.582 board)** | **53.6% (+1.0pp)** | **0.310 (+0.3pp)** | **0.506 (+2.1pp)** |

**LightGBM+class_weight dengan data gabungan adalah kandidat terbaik proyek
secara keseluruhan** — accuracy tertinggi (53.6%) DAN F1 macro/weighted
terbaik sekaligus, tidak ada trade-off seperti biasanya. XGBoost acc-tuned
accuracy-nya sedikit turun (kemungkinan karena hyperparameter di-tuning
khusus untuk distribusi 10.223 board yang lama; re-tuning di atas data baru
belum dicoba) tapi F1 macro/weighted-nya tetap naik.

**Proses mendapatkan data ini**:
1. Permintaan awal (crawler BBO skala besar) ditolak — ToS BBO eksplisit
   melarang scraping/automation software.
2. Dicek alternatif non-BBO: 2 dari 3 sumber yang muncul di hasil pencarian
   ternyata sudah mati (domain expired/di-squat) meski terlihat aktif di
   cache Google — pelajaran penting: **selalu verifikasi langsung, jangan
   percaya cuma dari snippet hasil pencarian**.
3. `computerbridge.se` terverifikasi hidup — 1.314 board dari 7 final
   kejuaraan dunia (Bermuda Bowl, Venice Cup, Vanderbilt, Spingold, dll.),
   format PBN, non-komersial.
4. **Ditemukan bonus**: 100 file LIN baru muncul di `data/raw/` selama
   sesi ini (kemungkinan besar export manual pribadi dari BBO — legitimate,
   beda dari scraping otomatis) — menambah ~2.472 board mentah lagi.
5. Dibangun `src/parser/pbn_parser.py` (PBNParser) supaya format PBN bisa
   masuk ke pipeline yang sama tanpa mengubah `extract_features()`.
   `build_dataset()` diperluas dengan parameter `extra_boards` untuk
   menggabungkan sumber manapun.

**Rekomendasi**: `data/processed_combined/` (606 file LIN + 43 file PBN,
182 fitur, 13.582 board) layak jadi dataset utama baru menggantikan
`data/processed_dds/` — LightGBM+class_weight di atasnya adalah kandidat
model terbaik proyek sejauh ini. Re-tuning XGBoost khusus untuk distribusi
data baru ini adalah langkah lanjutan yang masuk akal kalau mau
memaksimalkan lebih jauh.
